In [ ]:
from pathlib import Path # serve a lavorare con perrcorsi di file e cartelle in modo comodo

In [ ]:
path = Path("/Users/giorgiabrillanti/pneumonia-mortality-pipeline/data/raw/CF_dataset.txt")

In [ ]:
lines = path.read_text(encoding="utf-8").splitlines() # apre il file e lo legge come un unico grande testo
                                                      # splitlines spezza il testo in una lista di righe

In [ ]:
print(len(lines))

In [ ]:
print(set(line.count("\t") for line in lines)) # conta le tabulazioni in una riga, ripete il conteggio per ogni riga della lista
                                               # set raccoglie i risultati togliendo i doppioni

In [ ]:
ids = [] # creo una lista vuota dove inserire gli identificativi
records = [] # creo una lista vuota in cui inserire i valori dei pazienti

for line in lines:
    patients_id, record = line.split("\t")
    ids.append(patients_id)
    records.append(record)

print(ids[:3])
print(len(set(ids)))

In [ ]:
### guardo l'inizio dei dati
for r in records[:5]:
    print(r[:110])

In [ ]:
tokens = records[0].split("_")

In [ ]:
tokens

In [ ]:
tokens = records[0].split("_")  # splitto tutto dividendo con _
hospital, age, sex, temp, comorbidity, pcr, ct, morbidity = tokens[:8] # fino alla var 8 tutto bene
mortality = tokens[8].split(" ")[0] # per la var 8 invece c'è uno spazio in mezzo, prendo la prima parte dello split e lo metto dentro mortality

print(hospital, age, sex, temp, comorbidity, pcr, ct, morbidity, mortality)
print(tokens[8])
print(mortality)

In [ ]:
bad = 0
for r in records:
    t = r.split("_")
    if t[8].split(" ")[0] not in ("Cured", "Deceased", "Unknown"):
        bad += 1

print(bad)

In [ ]:
# controlliamo quanti pezzi ha ogni paziente
import pandas as pd

n_tokens = pd.Series([len(r.split("_")) for r in records])
print(n_tokens.describe())
print(n_tokens.value_counts().head(10))

Ogni record: 8 dati generali, poi 8 gruppi di esami (sempre presenti, anche vuoti) e ogni esame occupa 4 pezzi

In [ ]:
# 16 pezzi fissi: 8 dati generali + 8 nomi di gruppo
print(((n_tokens - 16) % 4 == 0).mean())

# ogni esame occupa 4 pezzi
n_tests = (n_tokens - 16) // 4
print(n_tests.describe())

In [ ]:
rows = []
for patient_id, record in zip(ids, records): # appaia paziente con i suoi dati
    t = record.split("_") # spezza il record a ogni trattino
    rows.append({
        "patient_id": patient_id,
        "hospital": t[0],
        "age": t[1],
        "sex": t[2],
        "temperature": t[3],
        "comorbidity": t[4],
        "pcr": t[5],
        "ct": t[6],
        "morbidity": t[7],
        "mortality": t[8].split(" ")[0],
    })

general = pd.DataFrame(rows)
general.head()

In [ ]:
print(general["mortality"].value_counts())
print(general["morbidity"].value_counts())
print(general["hospital"].value_counts())

In [ ]:
pd.crosstab(general["morbidity"], general["mortality"])

In [ ]:
general["age"] = pd.to_numeric(general["age"], errors="coerce")
general["temperature"] = pd.to_numeric(general["temperature"], errors="coerce")

print(general[["age", "temperature"]].describe())
print(general[["age", "temperature"]].isna().sum())

In [ ]:
raw = pd.DataFrame(rows) # dalle righe creo un dataset, ci serve per vedere i missing che in general non ci sono piu
print(raw.loc[general["temperature"].isna(), "temperature"].value_counts()) # .loc prende dal dataset solo le righe in cui la condizione è vera

print((general["age"] < 18).sum())

In [ ]:
pop = general[general["mortality"].isin(["Cured", "Deceased"])]
print(len(pop))
print((pop["age"] < 18).sum())
print(pop.loc[pop["age"] < 18, "mortality"].value_counts())

In [ ]:
GROUPS = [ # lista degli otto nomi di gruppo
    "Routine Blood Test", "Inflammation", "Blood Coagulation Test",
    "Biochemical Test", "Immune Cell Typing", "Cytokine Profile Test",
    "Autoimmune Test", "Routine Urine Test",
]


def parse_tests(record):
    tokens = [t.strip() for t in record.split("_")]
    rest = tokens[8:] # prende i pezzi dal nono in poi
    rest[0] = rest[0].split(" ", 1)[1]   # toglie l'esito, resta il nome del gruppo

    tests = []
    group = None
    i = 0
    while i < len(rest):
        if rest[i] in GROUPS:
            group = rest[i]
            i += 1
        else:
            name, abbr, value, ref = rest[i:i + 4] # prende un pezzo di lista, da i a i+3
            tests.append({"group": group, "name": name, "abbr": abbr,
                          "value": value, "ref": ref})
            i += 4
    return tests


tests_1 = parse_tests(records[0])
print(len(tests_1))
print(tests_1[:2])

In [ ]:
tokens = records[0].split("_")
rest = tokens[8:]
rest[0] = rest[0].split(" ", 1)[1]

print(len(rest))
print([(i, x) for i, x in enumerate(rest) if x in GROUPS])
print(rest[-6:])

In [ ]:
records[1:] # parto dal secondo elemento, fino alla fine

In [ ]:
type(records)

In [ ]:
len(records)

In [ ]:
type(record[0])

In [ ]:
records[0][:200]

In [ ]:
records[:2]

In [ ]:
all_tests = []
for patient_id, record in zip(ids, records):
    for test in parse_tests(record):
        test["patient_id"] = patient_id
        all_tests.append(test)

tests_long = pd.DataFrame(all_tests)
print(tests_long.shape)
print(n_tests.sum())
tests_long.head()

In [ ]:
dups = tests_long.duplicated(subset=["patient_id", "abbr"])
print(dups.sum())
print(tests_long.loc[dups, ["patient_id", "group", "name", "abbr"]].head())

In [ ]:
tests_wide = tests_long.pivot(index="patient_id", columns="abbr", values="value")
tests_wide = tests_wide.reset_index()

full = general.merge(tests_wide, on="patient_id", how="left")
print(tests_wide.shape)
print(full.shape)
full.iloc[:3, :14]

In [ ]:
from pathlib import Path

out = Path("../data/processed")
out.mkdir(parents=True, exist_ok=True)
full.to_csv(out / "cf_wide_text.csv", index=False)

In [ ]:
from pathlib import Path
print(Path.cwd())
print(out.resolve())